# Lektion 10 - KI-Agenten in der Produktion

In dieser Lektion lernen Sie **Produktionsmuster** für KI-Agenten unter Verwendung des Microsoft Agent Frameworks (Python). Wir behandeln:

- **Beobachtbarkeit** — Hinzufügen von Zeitmessung und Protokollierung zu Agenteninteraktionen
- **Bewertung** — Einsatz eines Bewertungsagenten zur Qualitätsbewertung von Antworten
- **Kostenmanagement** — Strategien zur Token-Optimierung und Modellauswahl

Das Szenario ist ein **Reisebüroagent**, der Nutzern bei der Planung von Reisen hilft, mit Überwachung und Bewertung als zusätzlicher Schicht.


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produktionsüberlegungen

Die Überführung von KI-Agenten von Prototypen in die Produktion erfordert sorgfältige Beachtung dreier Säulen:

1. **Beobachtbarkeit** — Sie benötigen Einblick darin, was der Agent tut, wie lange es dauert und welche Werkzeuge er aufruft. Ohne Tracing und Protokollierung ist die Fehlerbehebung bei Produktionsproblemen nahezu unmöglich.

2. **Evaluation** — Automatisierte Qualitätsprüfungen stellen sicher, dass die Antworten des Agenten im Laufe der Zeit genau, vollständig und hilfreich bleiben. Ein Bewertungsagent kann Antworten anhand definierter Kriterien bewerten.

3. **Kostenmanagement** — Die Token-Nutzung wirkt sich direkt auf die Kosten aus. Strategien wie Prompt-Optimierung, Modellauswahl und Caching helfen, die Ausgaben unter Kontrolle zu halten, ohne die Qualität zu beeinträchtigen.


## Aufbau eines Observable Agent

Wir definieren Reisetools und umhüllen den Agentenaufruf mit einer Zeitmessung, damit wir die Latenz überwachen können. In der Produktion würden Sie OpenTelemetry oder ein ähnliches Tracing-Backend integrieren.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Bewertungsmuster

Ein gängiges Produktionsmuster ist die Verwendung eines zweiten Agenten als **Evaluator**. Der Evaluator bewertet die Antwort des primären Agenten anhand vordefinierter Kriterien wie Vollständigkeit, Genauigkeit und Hilfsbereitschaft.

Dies ermöglicht:
- Automatisierte Qualitätssicherungen, bevor Antworten Benutzer erreichen
- Regressionserkennung, wenn sich Aufforderungen oder Modelle ändern
- Kontinuierliche Überwachung der Agentenleistung im Zeitverlauf


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategien zur Kostenkontrolle

Die Kontrolle der Kosten ist entscheidend für produktive KI-Agenten. Hier sind wichtige Strategien:

| Strategie | Beschreibung |
|---|---|
| **Prompt-Optimierung** | Halten Sie Systemanweisungen knapp. Entfernen Sie redundanten Kontext, um Eingabe-Token zu reduzieren. |
    "| **Modellauswahl** | Verwenden Sie kleinere, günstigere Modelle (z.B. GPT-4.1-mini) für einfache Aufgaben wie Klassifikation oder Extraktion und reservieren Sie größere Modelle für komplexes Denken. |\n",
| **Caching** | Speichern Sie Tool-Ergebnisse und häufige Anfragen zwischen, um redundante API-Aufrufe zu vermeiden. |
| **Token-Budgets** | Setzen Sie `max_tokens`-Grenzen, um unerwartet lange Antworten zu vermeiden. |
| **Batch-Verarbeitung** | Bündeln Sie mehrere Nutzeranfragen in einem einzigen API-Aufruf, wo möglich. |

In der Praxis funktioniert ein gestufter Ansatz gut: Leiten Sie einfache Anfragen an ein schnelles, kostengünstiges Modell weiter und eskalieren Sie nur komplexe Anfragen an ein leistungsfähigeres Modell.


## Zusammenfassung

In dieser Lektion hast du gelernt:

1. **Beobachtbarkeit** zu Agenteninteraktionen mit Timing und Protokollierung hinzuzufügen, und damit die Grundlage für Tracing und Monitoring zu legen.
2. **Agentenantworten zu bewerten** mithilfe eines Bewertungsagenten, der Vollständigkeit, Genauigkeit und Hilfreichkeit beurteilt.
3. **Kosten zu verwalten** durch Optimierung der Eingabeaufforderungen, Modellauswahl, Caching und Token-Budgets.

Diese Produktionsmuster helfen sicherzustellen, dass deine KI-Agenten zuverlässig, messbar und kosteneffizient im großen Maßstab sind.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
